# EDA on taxi trip pricing

In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns 

In [ ]:
df = pd.read_csv("../data/taxi_trip_pricing.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().sum().sort_values(ascending=False)

## Choosing the feautres and labels for the ML-prediction.

In [ ]:
selected_cols = [
    "Trip_Distance_km",
    "Trip_Duration_Minutes",
    "Time_of_Day",
    "Passenger_Count",
    "Trip_Price",
]

df_reduced = df[selected_cols]

## Some graphs on the new data for further analysis

In [ ]:
df_reduced.hist(figsize=(10,8))
plt.show()

In [ ]:
plt.figure()
plt.scatter(df_reduced["Passenger_Count"], df_reduced["Trip_Price"], alpha=0.7)
plt.title("Trip price vs Passenger amount")
plt.xlabel("Passenger Count")
plt.ylabel("Trip price")
plt.show()

In [ ]:
plt.figure()
plt.scatter(df_reduced["Trip_Distance_km"], df_reduced["Trip_Price"], alpha=0.7)
plt.title("Trip price vs Trip distance in km")
plt.xlabel("Trip Distance in KM")
plt.ylabel("Trip price")
plt.show()

In [ ]:
plt.figure()
plt.scatter(df_reduced["Trip_Duration_Minutes"], df_reduced["Trip_Price"], alpha=0.7)
plt.title("Trip price vs Trip duration in minutes")
plt.xlabel("Trip duration in minutes")
plt.ylabel("Trip price")
plt.show()

## NaN values analysis

### Label missing (Trip_Price)
- Will not be used for training
- Will be saved for predictions

In [ ]:
df_with_label = df_reduced[df_reduced["Trip_Price"].notna()].copy()
df_no_label = df_reduced[df_reduced["Trip_Price"].isna()].copy()

## Nulls in features & Categorical feature
Numerical features
- Trip_Distance_km
- Trip_Duration_Minutes
- Passenger count

Categorical feature
- Replace with the most frequent value

In [ ]:
num_cols = ["Trip_Distance_km", "Trip_Duration_Minutes", "Passenger_Count"]

for col in num_cols:
    df_with_label[col] = df_with_label[col].fillna(df_with_label[col].median())

df_with_label["Time_of_Day"] = df_with_label["Time_of_Day"].fillna(
    df_with_label["Time_of_Day"].mode(dropna=True)[0]
)


## Outliers 
- Focus: Label and some features but not all of them
- Leaving Time_of_Day out of it

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(6, 12))

sns.boxplot(x=df_with_label["Trip_Price"], ax=axes[0], showfliers=True)
sns.boxplot(x=df_with_label["Passenger_Count"], ax=axes[1], showfliers=True)
sns.boxplot(x=df_with_label["Trip_Distance_km"], ax=axes[2], showfliers=True)
sns.boxplot(x=df_with_label["Trip_Duration_Minutes"], ax=axes[3], showfliers=True)

plt.tight_layout()
plt.show()

## Remove high values and negative values. 
- Extreme values may twist the model

In [ ]:
df_with_label = df_with_label[
    (df_with_label["Trip_Price"] > 0) &
    (df_with_label["Trip_Price"] < 110)
    ]

df_with_label = df_with_label[
    (df_with_label["Passenger_Count"] > 0) &
    (df_with_label["Passenger_Count"] < 4)
    ]

df_with_label = df_with_label[
    (df_with_label["Trip_Duration_Minutes"] > 0) &
    (df_with_label["Trip_Duration_Minutes"] < 120)
    ]

df_with_label = df_with_label[
    (df_with_label["Trip_Distance_km"] > 0) &
    (df_with_label["Trip_Distance_km"] < 60)
    ]

In [ ]:
df_with_label.info()
df_with_label.isna().sum()

## Export cleaned data 

In [ ]:
df_with_label.to_csv("taxi_cleaned_training_data.csv", index=False)
df_no_label.to_csv("taxi_prediction_inputs.csv", index=False)